# DeepSORT Final Project — Colab

**Runtime:** GPU (T4). **Данные:** `MyDrive/Videos-CV/`

⚠️ Используйте **эту версию** ячейки 2. Если видите «Гotovo» без «Проверка OK» — у вас старый ноутбук.

In [ ]:
!nvidia-smi

In [ ]:
# === ЯЧЕЙКА 2: клон + установка (НЕ используйте pip install -r requirements.txt) ===
REPO_URL = "https://github.com/danvlak-batya/deep-sort-project.git"
PROJECT_DIR = "/content/deep-sort-project"

import os, sys
os.chdir("/content")
if os.path.exists(PROJECT_DIR):
    !rm -rf {PROJECT_DIR}
!git clone {REPO_URL} {PROJECT_DIR}
os.chdir(PROJECT_DIR)

# Проверка: в репозитории должна быть timm-версия ReID
assert os.path.exists("reid/timm_backend.py"), (
    "Старый код на GitHub! Сделайте git push с компьютера и перезапустите.")

# Ставим пакеты ПО ОДНОМУ (без requirements.txt — там могут быть проблемные пакеты)
pkgs = [
    "numpy>=1.22,<2.1", "opencv-python", "scipy", "filterpy",
    "PyYAML", "scikit-learn", "Pillow", "tqdm", "ultralytics", "timm",
]
for pkg in pkgs:
    print("Installing", pkg)
    !{sys.executable} -m pip install -q {pkg}
!{sys.executable} -m pip install -q -U timm

import ultralytics, timm, torch
print("\n=== Проверка OK ===")
print("torch:", torch.__version__)
print("timm:", timm.__version__)
print("ultralytics: OK")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/Videos-CV"
import os
assert os.path.isdir(DATA_ROOT)
print(os.listdir(DATA_ROOT))

In [ ]:
from utils.mot_paths import find_sequence_dir, list_image_filenames

SEQUENCE_FOLDER = "Kitti-17"
SEQUENCE = "KITTI-17"
DETECTOR = "yolov8n"
REID = "osnet_x0_25"

SEQ_DIR = find_sequence_dir(DATA_ROOT, SEQUENCE_FOLDER)
print("Кадров:", len(list_image_filenames(SEQ_DIR)))

In [ ]:
import os
os.makedirs("results/demo", exist_ok=True)

!python run_tracker.py \
  --sequence_dir "{SEQ_DIR}" \
  --config configs/default.yaml \
  --detector {DETECTOR} \
  --reid {REID} \
  --output_file results/demo/{SEQUENCE}.txt

p = f"results/demo/{SEQUENCE}.txt"
print("OK:", os.path.exists(p), "size:", os.path.getsize(p) if os.path.exists(p) else 0)

In [ ]:
!python tools/generate_overlays.py \
  --mot_dir "{DATA_ROOT}" \
  --results_dir results/demo \
  --output_dir results/overlays \
  --sequence {SEQUENCE}

from IPython.display import Video, display
v = f"results/overlays/{SEQUENCE}_overlay.mp4"
display(Video(v, embed=True)) if os.path.exists(v) else print("Нет видео")